# Geometric Binary Predicates: The Building Blocks of Geometric Queries
### by [Kate Vavra-Musser](https://vavramusser.github.io) for the [R Spatial Notebook Series](https://vavramusser.github.io/r-spatial)

## Introduction
Spatial analyses often begin with questions about relationships between geographic features: *Does one feature lie within another? Do features overlap or share a boundary? Does one feature entirely contain another?* These questions, collectively referred to as **geometric queries**, are foundational to many workflows in spatial data science and the social sciences.

In traditional GIS software like ArcGIS or QGIS, these relationships are often evaluated visually through tools like *Select by Location* or specific geoprocessing tools. In the R programming environment, we move beyond visual interpretation by using well-defined geometric functions. These functions allow for reproducible, automated, and scalable workflows, making them ideal for complex analyses or large datasets.

This notebook introduces three commonly used **geometric binary predicate** functions from the [**`sf`**](https://cran.r-project.org/web/packages/sf/index.html) package:

* *`st_contains`* checks if one feature completely contains another (inverse of *`st_within`*)
* *`st_within`* checks if one feature is entirely contained within another (inverse of *`st_contains`*)
* *`st_intersects`* checks if two features share any part of their space

### What is a Geometric Binary Predicate?
A **geometric binary predicate** is a function that takes two sets of geometric features (points, lines, or polygons) as input and returns a vector (list) of boolean value (true or false) based on the spatial relationship between the two feature sets.

* **Binary** – It operates on two geometric objects.
* **Geometric** – It is related to spatial relationships such as intersection, containment, adjacency, etc.
* **Predicate** – It evaluates a condition and returns a boolean (true or false) result.

Geometric binary predicates are crucial in spatial analysis, spatial databases, GIS, and computational geometry because they enable **efficient spatial reasoning, querying, and decision-making** based on **geometric relationships**.

**Examples of Geometric Binary Predicates**
* **Intersects(A, B)** → true if A and B share at least one point
* **Touches(A, B)** → true if A and B share a boundary but not their interior
* **Contains(A, B)** → true if A completely encloses B
* **Within(A, B)** → true if A is completely inside B
* **Disjoint(A, B)** → true if A and B have no points in common
* **Equals(A, B)** → true if A and B represent the same geometric shape

In R spatial analysis, particularly in social science applications, geometric binary predicates are **not typically used in isolation**. Instead, they are the building blocks for more advanced analyses such as **spatial joins**, **overlays**, **filtering operations**, and **proximity calculations**. These functions allow users to analyze relationships between spatial entities.

In later notebooks, we will explore how to incorporate these foundational spatial relationship functions into more complex workflows.

### Why Geometric Queries Matter for Social Science Workflows?
In the social sciences, spatial data often represent individuals, households, neighborhoods, or administrative units like census tracts. Understanding spatial relationships among these features can help answer questions such as:

* Are schools located within specific catchment zones?
* How many health clinics fall within high-poverty neighborhoods?
* Do city boundaries overlap with environmentally protected areas?

These types of questions can be addressed systematically and reproducibly using geometric queries, which abstract the spatial relationships into logical comparisons. Mastering these concepts is critical for transitioning from traditional GIS tools to programmatic workflows, where such functions are key to enabling scalable and efficient analyses.

### Notebook Goals
By the end of this notebook, you will be able to:

* Understand the purpose and functionality of the binary geometric query **`sf`** functions *`st_contains`*, *`st_intersects`*, and *`st_within`*.
* Apply these functions to evaluate spatial relationships between geographic features in R.
* Recognize how these functions fit into larger workflows, such as spatial joins and proximity analyses.
* Appreciate the broader set of geometric comparison functions available in the **`sf`** package.

### ✨ Prerequisites ✨
* Complete [Introduction to sf: Reading, Writing, and Inspecting Vector Data](https://platform.i-guide.io/notebooks/9968babe-22e4-4c3d-98e2-d8b45e9672cd)
* Complete [Working with CRS: Reprojection and Transformation](https://platform.i-guide.io/notebooks/76912ca7-73e4-437e-8ecf-0cb456bd7282)
* Complete [Preparing Vector Data for Analysis](https://platform.i-guide.io/notebooks/44926d85-7f08-4774-a103-a22ff3876cad)
* Complete [IPUMS NHGIS Data Extraction Using ipumsr: Supplemental Exercise 1](https://platform.i-guide.io/notebooks/a74fff96-4db5-430f-b346-958b0c5f7b38)
* Complete [IPUMS NHGIS Data Extraction Using ipumsr: Supplemental Exercise 2](https://platform.i-guide.io/notebooks/bc79eda6-8353-42ea-8cb7-5db70aa6febf)
* Complete [Importing Data from Natural Earth with rnaturalearth and rnaturalearthdata](https://platform.i-guide.io/notebooks/934e764c-d727-4bab-bc1c-198233484adc)

### 💽 Data Used in this Notebook 💽
* United States Populated Places Point Locations Shapefile (*ipums_nhgis_places.shp*)
  * If you worked through [IPUMS NHGIS Data Extraction Using ipumsr: Supplemental Exercise 1](https://platform.i-guide.io/notebooks/a74fff96-4db5-430f-b346-958b0c5f7b38) you should have created and saved a copy of *ipums_nhgis_places.shp* in the final section of the notebook.
  * You can also download a copy of *ipums_nhgis_places.zip* file from [the I-GUIDE platform](https://platform.i-guide.io/datasets/514a645c-fb7e-405e-99de-ac12cf1b143b) or directly from [Kate's GitHub](https://github.com/vavramusser/r-spatial/blob/main/ipums_nhgis_places.zip).  You will need to unzip *ipums_nhgis_places.zip* and extract *ipums_nhgis_places.shp* file to your workspace.
* United States State Boundaries Shapefile (*ipums_nhgis_states.shp*)
  * If you worked through [IPUMS NHGIS Data Extraction Using ipumsr: Supplemental Exercise 2](https://platform.i-guide.io/notebooks/bc79eda6-8353-42ea-8cb7-5db70aa6febf) you should have created and saved a copy of *ipums_nhgis_states.shp* in the final section of the notebook.
  * You can also download a copy of *ipums_nhgis_states.zip* file from [the I-GUIDE platform](https://platform.i-guide.io/datasets/1a5acd50-4741-447a-bf36-2331b39559af) or directly from [Kate's GitHub](https://github.com/vavramusser/r-spatial/blob/main/ipums_nhgis_states.zip).  You will need to unzip *ipums_nhgis_states.zip* and extract *ipums_nhgis_states.shp* file to your workspace.
* Time Zone Boundary Shapefile (*time_zones.shp*)
  * If you worked through [Importing Data from Natural Earth with rnaturalearth and rnaturalearthdata](https://platform.i-guide.io/notebooks/934e764c-d727-4bab-bc1c-198233484adc) you should have created and saved a copy of *time_zones.shp* in the final section of the notebook.
  * You can also download a copy of *time_zones.zip* file from [the I-GUIDE platform](https://platform.i-guide.io/datasets/b69f8825-7807-401f-b6d9-f7d8754adb07) or directly from [Kate's GitHub](https://github.com/vavramusser/r-spatial/blob/main/time_zones.zip).  You will need to unzip *time_zones.zip* and extract *time_zones.shp* file to your workspace.

### Notebook Overview
1. Setup
2. Exploring Geometric Binary Predicate Functions from the sf Package
3. A List of Geometric Binary Predicate Functions from the sf Package

---

## 1. Setup
This notebook requires the following R packages and functions.

##### Required Packages

[**sf**](https://cran.r-project.org/web/packages/sf/index.html) · Support for [simple features](https://r-spatial.github.io/sf/articles/sf1.html), a standardized way to encode spatial vector data - Binds to [*GDAL*](https://gdal.org/en/stable) for reading and writing data, to [*GEOS*](https://libgeos.org) for geometrical operations, and to [*PROJ*](https://proj.org/en/stable) for projection conversions and datum transformations - Uses by default the [*s2*](https://cran.r-project.org/web/packages/s2/index.html) package for spherical geometry operations on ellipsoidal (long/lat) coordinates · This notebook uses the following functions from *sf*.

* [*geos_binary_pred*](https://rdrr.io/cran/sf/man/geos_binary_pred.html) · geometric binary predicates on pairs of simple feature geometry sets
  * *st_contains*
  * *st_intersects*
  * *st_within*
* [*st_crs*](https://rdrr.io/cran/sf/man/st_crs.html) · retrieve coordinate reference system from object
* [*st_read*](https://rdrr.io/cran/sf/man/st_read.html) · read simple features or layers from file or database
* [*st_transform*](https://rdrr.io/cran/sf/man/st_transform.html) · transform or convert coordinates of simple feature
* [*valid*](https://rdrr.io/cran/sf/man/valid.html) · check validity or make an invalid geometry valid
  * *st_make_valid* · make an invalid geometry valid
  * *st_is_valid* · check validity

### 1a. Install and Load Required Packages
If you have not already installed the required packages, uncomment and run the code below:

In [1]:
# install.packages("sf")

Load the packages into your workspace.

In [2]:
library(sf)

Linking to GEOS 3.9.1, GDAL 3.2.1, PROJ 7.2.1; sf_use_s2() is TRUE



### 1b. Import Data

First we will import three vector datasets to work with using the [*st_read*](https://rdrr.io/cran/sf/man/st_read.html) function from the [**sf**](https://cran.r-project.org/web/packages/sf/index.html) package.  If you worked through [Chapter 3.4.1](), [Chapter 3.4.2](), and [Chapter 4.1](), you should have saved these files to your workspace and they should be available to work with directly.  If you did not work through these chapters, you can download and import the files to your workspace.

* **United States Populated Places Point Locations** (*ipums_nhgis_places.shp*) downloaded from [IPUMS NHGIS](https://www.nhgis.org) in [Chapter 3.4.1 IPUMS NHGIS Data Extraction Using ipumsr - Supplemental Exercise 1](). 
* **United States State Boundaries** (*ipums_nhgis_states.shp*) downloaded from [IPUMS NHGIS](https://www.nhgis.org) in [Chapter 3.4.2 IPUMS NHGIS Data Extraction Using ipumsr - Supplemental Exercise 2](). 
* **Global Time Zone Boundaries** (*time_zones.shp*) downloaded from [Natural Earth](https://www.naturalearthdata.com) in [Chapter 4.1 Importing Data from Natural Earth with rnaturalearth and rnaturalearthdata](). 

In [8]:
unzip("ipums_nhgis_places.zip")
unzip("ipums_nhgis_states.zip")
unzip("time_zones.zip")

places <- st_read("ipums_nhgis_places.shp")
states <- st_read("ipums_nhgis_states.shp")
time_zones <- st_read("time_zones.shp")

Reading layer `ipums_nhgis_places' from data source 
  `C:\Users\vavra\Dropbox\R Spatial\r-spatial\07 Foundational Spatial Analysis\ipums_nhgis_places.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 29261 features and 8 fields
Geometry type: POINT
Dimension:     XY
Bounding box:  xmin: -6238595 ymin: -1328241 xmax: 2254129 ymax: 4577031
Projected CRS: USA_Contiguous_Albers_Equal_Area_Conic
Reading layer `ipums_nhgis_states' from data source 
  `C:\Users\vavra\Dropbox\R Spatial\r-spatial\07 Foundational Spatial Analysis\ipums_nhgis_states.shp' 
  using driver `ESRI Shapefile'
Simple feature collection with 52 features and 4 fields
Geometry type: MULTIPOLYGON
Dimension:     XY
Bounding box:  xmin: -7115208 ymin: -1685018 xmax: 3321632 ymax: 4591848
Projected CRS: USA_Contiguous_Albers_Equal_Area_Conic
Reading layer `time_zones' from data source 
  `C:\Users\vavra\Dropbox\R Spatial\r-spatial\07 Foundational Spatial Analysis\time_zones.shp' 
  using driver `ESRI Shapef

We can see that the *places* file contains point locations and information on 29,261 populated places in the US, the *states* file contains boundaries and information on 52 US states (50 states, Washington DC, and Puerto Rico), and the *time_zone* file contains boundaries and information on 120 time global time zones.

### 1c. Project All Files to the Same CRS

Let's ensure that all three files are using the same coordinate reference system (CRS).  Based on the file information output from reading in the data in the above code chunk, we can see that the two files from [IPUMS NHGIS](https://www.nhgis.org) (*places* and *states*) both use the [USA Contiguous Albers Equal Area Conic projected CRS](https://epsg.io/102003) while the *time_zones* file from [Natural Earth](https://www.naturalearthdata.com) uses the [WGS 1984 Geodetic CRS](https://epsg.io/4326).

For this notebook, we will use the [WGS 1984 Geodetic CRS](https://epsg.io/4326) used by *time_zones* so we will need to reproject the *places* and *states* files to this CRS.

In [9]:
places <- st_transform(places, st_crs(time_zones))
states <- st_transform(states, st_crs(time_zones))

Let's double check the CRS for all three files.

In [10]:
st_crs(places)$epsg
st_crs(states)$epsg
st_crs(time_zones)$epsg

[1] 4326

[1] 4326

[1] 4326

All three files are now represented in the [WGS 1984 Geodetic CRS (EPSG code 4326)](https://epsg.io/4326).

### 1d. Validate the Data

Let's ensure that the three files we are working with have valid geometries using the [*st_is_valid*](https://rdrr.io/cran/sf/man/valid.html) function from the [**sf**](https://cran.r-project.org/web/packages/sf/index.html) package.  If there are any invalid geometries that could impact our ability to carry out spatial analyses or properly map our data.

In [11]:
table(st_is_valid(places))


 TRUE 
29261 

In [12]:
table(st_is_valid(states))


FALSE  TRUE 
   12    40 

In [13]:
table(st_is_valid(time_zones))

Warning message in st_is_longlat(x):
"bounding box has potentially an invalid value range for longlat data"
Warning message in st_is_longlat(x):
"bounding box has potentially an invalid value range for longlat data"



FALSE  TRUE 
    2   118 

All 29,261 features in the *places* file are all valid.  However, 12 of the 52 features in the *states* file and 2 of the 120 features in the *time_zones* file are invalid.  We will need to validate these files before continuing using the [*st_make_valid*](https://rdrr.io/cran/sf/man/valid.html) function from the [**sf**](https://cran.r-project.org/web/packages/sf/index.html) package.

In [14]:
states <- st_make_valid(states)
time_zones <- st_make_valid(time_zones)

Warning message in st_is_longlat(x):
"bounding box has potentially an invalid value range for longlat data"


Let's check to see if the *states* and *time_zones* files have been completely validated.

In [15]:
table(st_is_valid(states))


TRUE 
  52 

In [16]:
table(st_is_valid(time_zones))


TRUE 
 120 

All 52 features inthe *states* file and 120 features in the *time_zones* file are now validated!  We are ready to move on to the next step.

### 1e. Review the Data

Before jumping into analysis, let's take a quick look at the first few rows for the *places* and *states* files as a refresher to see what attibutes (columns) each file includes.

In [17]:
head(places, 3)

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

,GISJOIN,PLACE,STATE,CL8AA1990,CL8AA2000,CL8AA2010,CL8AA2020,NAME,geometry
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<POINT [°]>
1,G01000100,Abanda CDP,Alabama,177.25,131.68,192,133.00,Abanda,POINT (-85.52968 33.10095)
2,G01000124,Abbeville city,Alabama,3176.41,2987.18,2688,2358.00,Abbeville,POINT (-85.25049 31.57184)
3,G01000460,Adamsville city,Alabama,5620.12,5107.32,4522,4353.93,Adamsville,POINT (-86.95611 33.60094)


In [18]:
head(states, 3)

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

,STATEFP10,STUSPS10,NAME10,REGION10,geometry
,<chr>,<chr>,<chr>,<chr>,<MULTIPOLYGON [°]>
1,56,WY,Wyoming,4,MULTIPOLYGON (((-104.058 44...
2,42,PA,Pennsylvania,1,MULTIPOLYGON (((-75.27089 3...
3,39,OH,Ohio,2,MULTIPOLYGON (((-83.00776 4...


A few attributes we may want to take note of for our analyses are:

From the ***places*** file:

* State Name (*STATE_x*)
* City Name (*PLACE_x*)
* 1990 Total Population (*CL8AA1990*)
* 2000 Total Population (*CL8AA2000*)
* 2010 Total Population (*CL8AA2010*)
* 2020 Total Population (*CL8AA2020*)

From the ***states*** file:

* State Name (*NAME10*)
* State USPS Abbreviation (*STUSPS10*)

## 2. Exploring Geometric Binary Predicate Functions from the sf Package

In this section we will explore [*binary predicate functions*](https://rdrr.io/cran/sf/man/geos_binary_pred.html) by stepping through the process of using them in practical examples.

The [*binary predicate functions*](https://rdrr.io/cran/sf/man/geos_binary_pred.html), (including [`st_contains`](https://rdrr.io/cran/sf/man/geos_binary_pred.html), [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html), [`st_within`](https://rdrr.io/cran/sf/man/geos_binary_pred.html), and others) will return an interesting type of data object called a **sparse geometry binary predicate (*sgbp*)** which represents the relationships between the two sets of features.  An *sgbp* is essentailly a list of feature index values (from the first set of features) each with a corresponding list of feature index values (from the second set of features) - a list of lists.  That can be a confusing data structure to imagine, but stepping through this section will help you to understand and visualize the structure of this type of data and how it is essential for spatial analyses.

An *sgbp* may seem like a convoluded and abstract way of representing the relationship between two sets of features but it is a very simple and lightweight way for a computer to store and understand a large set of relationships.  For the most part, you, the user, will not need to directly interact with *sgbp* objects because you likely will not have a reason to work with any of the [*binary predicate functions*](https://rdrr.io/cran/sf/man/geos_binary_pred.html) by themselves.  Instead, you will use these functions, and their output, in conjunction with key spatial query functions including spatial subsetting and joining ([`st_join`](https://rdrr.io/cran/sf/man/st_join.html), [`st_filter`](https://rdrr.io/cran/sf/man/st_join.html) and [`st_crop`](https://rdrr.io/cran/sf/man/st_crs.html)), spatial aggregation and geometry manipulation ([`st_union`](https://rdrr.io/cran/sf/man/geos_combine.html) and [`st_combine`](https://rdrr.io/cran/sf/man/geos_combine.html)), geometric binary operations ([`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_ops.html) and [`st_difference`](https://rdrr.io/cran/sf/man/geos_binary_ops.html)), spatial relationships ([`st_relate`](https://rdrr.io/cran/sf/man/st_read.html)), and others.

Although you may not need to work with the output to the [*binary predicate functions*](https://rdrr.io/cran/sf/man/geos_binary_pred.html), it is a good idea to have some understanding of how they work since they are an essential underlying part of working with and manipulating spatial data.

The next three sections step you through using [`st_contains`](https://rdrr.io/cran/sf/man/geos_binary_pred.html), [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html), and [`st_within`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) in practical examples.  For these sections, focus on understanding the relationship between the input pair of features and the resulting *sgbp* object as well as the structure of the *sgbp* object itself.

### 2a. Using [`st_contains`](https://rdrr.io/cran/sf/man/geos_binary_pred.html): Which States Contain which Places?

In this first exercise, we will use the [`st_contains`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function from the [`sf`](https://cran.r-project.org/web/packages/sf/index.html) package to explore which US states **contain** which populated places.

The following step will return a list of state index values each with a corresponding list of  populated place index values.

In [19]:
states_contains_places <- st_contains(states, places)

Let's take a look at the dimensions and the class of the resulting object.

In [20]:
dim(states_contains_places)
class(states_contains_places)

[1]    52 29261

[1] "sgbp" "list"

The *states_contains_places* object returned by the [`st_contains`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function is an *sgbp* object with **52** items (the index values for the 52 features in the *states* object) with attached *lists* containing **29,261** items (the index values for the 29,621 features in the *places* object).  In otherwords, the function returned a **list of state indicies** each with a corresponding **list of place indicies** representing the **states** and which **places** they contain.

Let's take a quick look at the first five items in the *states_contains_places* object.  This will show us the first five *states* and their corresponding lists of *places*.

In [21]:
states_contains_places[1:5]

[[1]]
  [1] 29059 29060 29061 29062 29063 29064 29065 29066 29067 29068 29069 29070
 [13] 29071 29072 29073 29074 29075 29076 29077 29078 29079 29080 29081 29082
 [25] 29083 29084 29085 29086 29087 29088 29089 29090 29091 29092 29093 29094
 [37] 29095 29096 29097 29098 29099 29100 29101 29102 29103 29104 29105 29106
 [49] 29107 29108 29109 29110 29111 29112 29113 29114 29115 29116 29117 29118
 [61] 29119 29120 29121 29122 29123 29124 29125 29126 29127 29128 29129 29130
 [73] 29131 29132 29133 29134 29135 29136 29137 29138 29139 29140 29141 29142
 [85] 29143 29144 29145 29146 29147 29148 29149 29150 29151 29152 29153 29154
 [97] 29155 29156 29157 29158 29159 29160 29161 29162 29163 29164 29165 29166
[109] 29167 29168 29169 29170 29171 29172 29173 29174 29175 29176 29177 29178
[121] 29179 29180 29181 29182 29183 29184 29185 29186 29187 29188 29189 29190
[133] 29191 29192 29193 29194 29195 29196 29197 29198 29199 29200 29201 29202
[145] 29203 29204 29205 29206 29207 29208 29209 29210 29211 29212 29213 29214
[157] 29215 29216 29217 29218 29219 29220 29221 29222 29223 29224 29225 29226
[169] 29227 29228 29229 29230 29231 29232 29233 29234 29235 29236 29237 29238
[181] 29239 29240 29241 29242 29243 29244 29245 29246 29247 29248 29249 29250
[193] 29251 29252 29253 29254 29255 29256 29257 29258 29259 29260 29261

[[2]]
   [1] 19960 21458 21459 21460 21461 21462 21463 21464 21465 21466 21467 21468
  [13] 21469 21470 21471 21472 21473 21474 21475 21476 21477 21478 21479 21480
  [25] 21481 21482 21483 21484 21485 21486 21487 21488 21489 21490 21491 21492
  [37] 21493 21494 21495 21496 21497 21498 21499 21500 21501 21502 21503 21504
  [49] 21505 21506 21507 21508 21509 21510 21511 21512 21513 21514 21515 21516
  [61] 21517 21518 21519 21520 21521 21522 21523 21524 21525 21526 21527 21528
  [73] 21529 21530 21531 21532 21533 21534 21535 21536 21537 21538 21539 21540
  [85] 21541 21542 21543 21544 21545 21546 21547 21548 21549 21550 21551 21552
  [97] 21553 21554 21555 21556 21557 21558 21559 21560 21561 21562 21563 21564
 [109] 21565 21566 21567 21568 21569 21570 21571 21572 21573 21574 21575 21576
 [121] 21577 21578 21579 21580 21581 21582 21583 21584 21585 21586 21587 21588
 [133] 21589 21590 21591 21592 21593 21594 21595 21596 21597 21598 21599 21600
 [145] 21601 21602 21603 21604 21605 21606 21607 21608 21609 21610 21611 21612
 [157] 21613 21614 21615 21616 21617 21618 21619 21620 21621 21622 21623 21624
 [169] 21625 21626 21627 21628 21629 21630 21631 21632 21633 21634 21635 21636
 [181] 21637 21638 21639 21640 21641 21642 21643 21644 21645 21646 21647 21648
 [193] 21649 21650 21651 21652 21653 21654 21655 21656 21657 21658 21659 21660
 [205] 21661 21662 21663 21664 21665 21666 21667 21668 21669 21670 21671 21672
 [217] 21673 21674 21675 21676 21677 21678 21679 21680 21681 21682 21683 21684
 [229] 21685 21686 21687 21688 21689 21690 21691 21692 21693 21694 21695 21696
 [241] 21697 21698 21699 21700 21701 21702 21703 21704 21705 21706 21707 21708
 [253] 21709 21710 21711 21712 21713 21714 21715 21716 21717 21718 21719 21720
 [265] 21721 21722 21723 21724 21725 21726 21727 21728 21729 21730 21731 21732
 [277] 21733 21734 21735 21736 21737 21738 21739 21740 21741 21742 21743 21744
 [289] 21745 21746 21747 21748 21749 21750 21751 21752 21753 21754 21755 21756
 [301] 21757 21758 21759 21760 21761 21762 21763 21764 21765 21766 21767 21768
 [313] 21769 21770 21771 21772 21773 21774 21775 21776 21777 21778 21779 21780
 [325] 21781 21782 21783 21784 21785 21786 21787 21788 21789 21790 21791 21792
 [337] 21793 21794 21795 21796 21797 21798 21799 21800 21801 21802 21803 21804
 [349] 21805 21806 21807 21808 21809 21810 21811 21812 21813 21814 21815 21816
 [361] 21817 21818 21819 21820 21821 21822 21823 21824 21825 21826 21827 21828
 [373] 21829 21830 21831 21832 21833 21834 21835 21836 21837 21838 21839 21840
 [385] 21841 21842 21843 21844 21845 21846 21847 21848 21849 21850 21851 21852
 [397] 21853 21854 21855 21856 21857 21858 21859 21860 21861

Let's take a quick look at the length of each list of place indicies for the states using the *lengths* function.  The output will show us number of places contained in each of the states.

In [22]:
lengths(states_contains_places)

[1]  203 1765 1201  443  518   34  377    0  773  401  129  624 1188  542  671
[16]  580  326  343  362  733  401  692  458  545   75  364  628  142 1522  524
[31]  244  912  227 1031  151  579  395   96  390 1367  427  682 1009  451  906
[46]  473    1  591 1751  119  131  738

From this we can see that the first state contains 203 places, the second state contains 1765 places, and so on.

You may notice that there is one state with 0 places and one state with only 1 place which seems out of the ordinary compared to the other states.  The state with 0 places is Puerto Rico which, of course, in reality does have populated places, but did not match to any features in the *places* object because the *places* object does not include information for Puerto Rico.  The state with 1 place is the District of Columbia which contains the single city of Washington DC.

Let's take a look at the summary information for the counts of places in each state.

In [23]:
summary(lengths(states_contains_places))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
    0.0   305.5   465.5   562.2   702.2  1765.0 

From this output we can see that states contain a mean and median of about 562 and 466 populated places, respectively.

### 2b. Using [`st_within`](https://rdrr.io/cran/sf/man/geos_binary_pred.html): Which Places are Within which States?

For the second exercise, we will use the [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function from the [`sf`](https://cran.r-project.org/web/packages/sf/index.html) package to explore which places are within each state.

As you might guess, this analysis is very similar to the one carried out in section 2a.  However, this time the results will be a list of *places* with corresponding lists of *states* rather than *states* with corresponding lists of *places*.  The [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function is the **inverse** of the [`st_within`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function and vice versa.

The following step will return a list of populated place index values each with a corresponding list of state index values.

In [24]:
places_within_states <- st_within(places, states)

Let's take a look at the dimensions of the resulting object.

In [25]:
dim(places_within_states)

[1] 29261    52

This time the result is an *sgbp* object with **29,261** items (the index values for the 29,621 features in the *places* object) with attached *lists* containing **52** items (the index values for the 52 features in the *states* object) - a **list of place indicies** each with a corresponding **list of state indicies**  representing the **places** and which **states** they are within.

Let's take a quick look at the first ten items in the *places_within_states* object.  

In [26]:
places_within_states[1:10]

[[1]]
[1] 36

[[2]]
[1] 36

[[3]]
[1] 36

[[4]]
[1] 36

[[5]]
[1] 36

[[6]]
[1] 36

[[7]]
[1] 36

[[8]]
[1] 36

[[9]]
[1] 36

[[10]]
[1] 36

Each of these first 10 places are in state 36 (Alabama) and exploring the *places_within_states* object will show you that each *place* corresponds to only one *state*.  The point geometries representing the point locations of each populated place can only be located within one polygon.

If you are used to thinking of relationships in terms of *one-to-one*, *one-to-many*, *many-to-one*, and *many-to-many* the results of this exercise is a *many-to-one* relationship (many places can be within a state) while the results of the previous exercise exploring which states contain which places was a *one-to-many* (a state can contain many places) relationship.

If our goal was to explore this same relationship but for metropolitain areas (which may span more than one state) we would need to use polygons rather that points to represent populated places .

### 2c. Using [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html): Which Timezones Intersect with which States?

For the final exercise, we will use the [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) function from the [`sf`](https://cran.r-project.org/web/packages/sf/index.html) package to explore which time zones intersect with which states.  Unlike the previous two examples which used a set of points and a set of polygons, this exercise uses two sets of polygons.

The following step will return a list of time zone index values each with a corresponding list of state index values.

In [27]:
timezones_intersects_states <- st_intersects(time_zones, states)

Let's take a look at the dimensions of the resulting object.

The *timezones_intersects_states* object will contain a **list of time zone indicies** each with a corresponding **list of state indicies** representing the states which intersect with the time zone.

Let's take a look at the *places_within_states* object.  

In [28]:
timezones_intersects_states

Sparse geometry binary predicate list of length 120, where the
predicate was `intersects'
first 10 elements:
 1: (empty)
 2: 18
 3: (empty)
 4: (empty)
 5: (empty)
 6: (empty)
 7: 18, 35
 8: (empty)
 9: (empty)
 10: 1, 4, 7, 10, 11, 15, 16, 17, 20, 23, ...

From this output we can see that the time zones with indicies 10, 12, 13, and 21 intersect with most of the states and probably represent the four main time zones used in the contiguious United States.  Time zones 2, 7, 17, 79 also intersect with US states.

We can also see that many of the time zones have no corresponding states - their list of states is empty.  This is because most of the global time zones do not cross the United States and do not contain any US states.  Recall from the introduction that *sgbp* stands for **sparse geometry binary predicate**.  In this case **sparse** refers to the fact that some of the elements in the object can be 0 (empty).  If you are familiar with the concept of a **sparse matrix** a **sparse geometry set** is an application of that concept.

An important thing to note here is that **some of the time zones intersect with more than one state** - some state indicies are in the lists for more than one time zone.  For example, state 18 (Alaska) is in the list for time zones 2, 7, and 79 while state 4 (New Mexico) is in the list for zones 10 and 12.  Again, if you are used to thinking of relationships in terms of *one-to-one*, *one-to-many*, *many-to-one*, and *many-to-many* the results of this exercise is a *many-to-many* relationship (a time zone can intersect with many states *and* a state can intersect with many time zones).  [`st_intersects`](https://rdrr.io/cran/sf/man/geos_binary_pred.html) and the other geometric binary predicate functions can produce *many-to-many* relationships.  It is important to have a clear understanding of the type of geometric binary predicate you're using and ensure it both fits your data and the analysis you are trying to carry out.

Let's take a look at the UTC format (*utc_format*) and places (*places*) for the four time zones which intersect with most of the states, zones 10, 12, 13, and 21.

In [29]:
time_zones[c(10, 12, 13, 21), c("utc_format", "places")]

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

,utc_format,places,geometry
,<chr>,<chr>,<MULTIPOLYGON [°]>
10,UTC-07:00,"Canada (northeastern British Columbia), Mexico (Sonora), United States (Arizona)",MULTIPOLYGON (((-97.86575 8...
12,UTC-06:00,"Canada (almost all of Saskatchewan), Costa Rica, El Salvador, Ecuador (Galapagos Islands), Guatemala, Honduras, Mexico (most), Nicaragua,",MULTIPOLYGON (((-82.50428 8...
13,UTC-05:00,"Colombia, Cuba, Ecuador (continental), Jamaica, Panama, Peru",MULTIPOLYGON (((-67.50302 8...
21,UTC-08:00,"Canada (most of British Columbia), Mexico (Baja California), United States (California, most of Nevada, most of Oregon, Washington (state))",MULTIPOLYGON (((-140.9369 6...


We can confirm that these four time zones correspond to the UTC-5 (Eastern), UTC-6 (Central), UTC-7 (Mountain), and UTC-8 (Pacific) time zones used in the contiguious United States.

Let's take a look at the other four zones which intersect US states, zones 2, 7, 17, and 79.

In [30]:
time_zones[c(2, 7, 17, 79), c("utc_format", "places")]

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

,utc_format,places,geometry
,<chr>,<chr>,<MULTIPOLYGON [°]>
2,UTC-10:00,United States (Aleutian Islands),MULTIPOLYGON (((179.7279 59...
7,UTC-10:00,United States (Hawaii),MULTIPOLYGON (((-169.3395 5...
17,UTC-04:00,"Bolivia, Brazil (Amazonas), Chile (continental), Dominican Republic, Canada (Nova Scotia), Puerto Rico, Trinidad and Tobago",MULTIPOLYGON (((-55.95404 4...
79,UTC-09:00,United States (most of Alaska),MULTIPOLYGON (((-127.5151 7...


These four include UTC-9 which is used by most of Alaska, two versions of UTC-10 which are used by Hawaii and the Aleutian Islands in Alaska, and UTC-4 which is used by Puerto Rico.

## 3. A List of Geometric Binary Predicate Functions from the [**sf**](https://cran.r-project.org/web/packages/sf/index.html) Package

Each of the [geometric binary predicate functions](https://rdrr.io/cran/sf/man/geos_binary_pred.html) takes two vector sf objects and returns a sparse geometry binary predicate (*sgbp*) object representing the relationships between the two sets of features.  Reading through this list you'll notice that the differences between many of the functions are very nuanced.  If you are new to [**sf**](https://cran.r-project.org/web/packages/sf/index.html) and the concept of spatial relationships in R, you can probably just focus on `st_contains`, `st_within`, `st_intersects`, and `st_touches` which will likely cover most of the geometric queries you will need in a typical analysis.
   
| Function               | Description | Importance |
|:------------------------|:-------------| :-------------|
| **st_contains**        | checks if one geometry completely contains another (inverse of `st_within`) | ✅ most important |
| **st_contains_properly** | like `st_contains`, but boundaries must not touch | ⚠️ learn later |
| **st_covered_by**      | checks if a geometry is fully inside another, including the boundary (inverse of `st_covers`) | ⚠️ learn later |
| **st_covers**          | checks if one geometry fully covers another, including the boundary (inverse of `st_covered_by`) | ⚠️ learn later |
| **st_crosses**         | checks if two geometries intersect but do not fully contain one another | ⛔️ advanced use |
| **st_disjoint**        | checks if two geometries have no points in common (inverse of `st_intersects`) | ⛔️ advanced use |
| **st_equals**          | checks if two geometries are exactly the same | ⛔️ advanced use |
| **st_equals_exact**    | like `st_equals`, but allows for small numerical differences | ⛔️ advanced use |
| **st_intersects**      | checks if two geometries share any space (inverse of `st_disjoint`) | ✅ most important |
| **st_is_within_distance** | checks if two geometries are within a specified distance | ⚠️ learn later |
| **st_overlaps**        | checks if two geometries share some space but are not fully contained within each other | ⛔️ advanced use |
| **st_touches**         | checks if two geometries share a boundary but do not overlap | ✅ most important |
| **st_within**          | checks if one geometry is entirely inside another (inverse of `st_contains`) | ✅ most important |

The *geometric binary predicate* functions are used in:

* **Spatial Joins and Spatial Filtering**
* **Overlay Operations**
* **Distance-Based Analysis**